# 00 — Start here

**The question:** *I have never called this API. Is it fresh enough to answer a
question about last month, what am I allowed to ask for, and how will it tell me
when I have asked for too much?*

Everything else in `notebooks/` assumes you have run this one. By the end you
will have called `catalog()`, `coverage()` and `metric_coverage()`, seen both
auth tiers, and deliberately triggered each of the three errors that matter.

SILO is a **read-only** API over Brazilian public filings — CVM funds, listed
companies, ANBIMA class aggregates, and B3 quotes, options and termo. There is
no ingest endpoint, and the landing tables are closed to every caller.

## Install

The SDK lives in this repository and is **not on PyPI**:

```bash
git clone https://github.com/PedroDnT/SILO-BZ
cd SILO-BZ
pip install -e sdk/          # add [pandas] for the wide panel helper
```

You can also call the API with nothing but `curl` or `httpx` — it is plain
PostgREST. The SDK is worth it for one reason: **it raises rather than hand you
a short series**. That is the whole point of it.

In [ ]:
# The SDK is not on PyPI. From the repository root:
#
#     pip install -e sdk/
#
# Auth is the shared publishable key printed in the docs. It is for TESTING:
# everyone reading the docs has the same one, so it identifies the project and
# not you. It puts you on the ANONYMOUS tier. Set SILO_TOKEN to a GitHub
# sign-in token (notebook 00) to run signed in.
import os

os.environ.setdefault("SILO_URL", "https://zcjbtpxuhdekpwcxmepn.supabase.co")
os.environ.setdefault(
    "SILO_ANON_KEY", "sb_publishable__yfFQsykAglrvc9GS6_PYw_B24ex437"
)

import pandas as pd

from silo_client import SiloClient

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

silo = SiloClient()
print(f"tier            : {silo.tier}")
print(f"catalog version : {silo.catalog()['version']}")

## The first call is always `coverage`

Not `panel`. Not `quote_history`. **`coverage`.**

Every number this API serves is only as current as the filing behind it, and
CVM publishes with a one-to-two month lag. A notebook that charts fund NAV
without checking coverage first will happily draw a line that stops in July and
looks like a line that ends in July.

This helper is re-used by every other notebook in this folder.

In [ ]:
def as_of(*datasets: str) -> pd.DataFrame:
    """Print how fresh every dataset this notebook relies on actually is.

    Run this FIRST, every time. A stale warehouse then shows up in the output
    instead of being silently baked into a number further down.

      as_of            the newest period that has landed AND has elapsed.
                       This is freshness.
      complete_through the newest period classified COMPLETE — what the
                       default windows serve.
      newest_period    the newest period KEY. It can sit in the FUTURE when a
                       family files forward-dated (FIP is keyed 31-December).
                       Never read this as freshness.
      landed_at        when ingest last SUCCEEDED. A later failed run never
                       advances it.
      notes            a caveat the dates cannot carry. Printed in full below,
                       never summarised, never dropped.
    """
    cov = pd.DataFrame(silo.coverage())
    rows = cov[cov["dataset"].isin(datasets)].copy()
    missing = set(datasets) - set(rows["dataset"])
    if missing:
        raise RuntimeError(f"coverage() has no row for {sorted(missing)}")
    print(
        rows[
            ["dataset", "as_of", "complete_through", "newest_period", "landed_at"]
        ].to_string(index=False)
    )
    for r in rows.itertuples():
        if r.notes:
            print(f"\n  CAVEAT [{r.dataset}]\n  {r.notes}")
    return rows.set_index("dataset")

In [ ]:
# Every dataset, so you can see the whole warehouse at once.
cov = pd.DataFrame(silo.coverage())
cov[["dataset", "as_of", "complete_through", "newest_period", "landed_at"]]

### Four dates, and only two of them are freshness

Read the table above carefully — the `funds_fip` row is the reason this matters.

FIP funds file **annually, keyed to 31-December**. So `newest_period` for that
family is a year-end date that has **not happened yet**, while `as_of` — the
newest period that has actually elapsed — is a year earlier. Reading
`newest_period` as freshness would have you claiming the API holds data from
the future.

In [ ]:
fip = cov.loc[cov["dataset"] == "funds_fip"].iloc[0]
print(f"funds_fip.newest_period   {fip['newest_period']}   <- a KEY, can be in the future")
print(f"funds_fip.as_of           {fip['as_of']}   <- freshness")
print(f"funds_fip.complete_through {fip['complete_through']}  <- what default windows serve")
print(f"funds_fip.landed_at       {fip['landed_at']}  <- our pipeline, not CVM's calendar")
print()
print("notes:", fip["notes"])

### `notes` carries what a date cannot

A `notes` value is a **regime boundary**: somewhere in that dataset a column
changes meaning at a date, and differencing across it produces a number that is
arithmetically fine and factually nonsense. Print it beside anything you compute
from that dataset.

In [ ]:
for r in cov.loc[cov["notes"].notna()].itertuples():
    print(f"[{r.dataset}]\n  {r.notes}\n")

## `catalog()` — the contract as JSON

`coverage()` says how fresh the data is. `catalog()` says what the rules are.
Call it once, cache it (the SDK does), and read the limits off the payload
instead of remembering them. `version` is the contract the database is actually
serving, which is not necessarily the one the docs site was written for — the
SDK warns (`SiloCatalogDrift`) when the two differ.

In [ ]:
cat = silo.catalog()
print("version   :", cat["version"])
print("primitive :", cat["primitive"])
print("metrics   :", ", ".join(sorted(cat["metrics"])))
print()
print("functions and views this server actually publishes:")
for name, route in sorted(cat["postgrest"].items()):
    print(f"  {name:22s} {route}")

### Metric names come from here, never from memory

This is worth being blunt about, because the failure is silent:

> **An unrecognised metric name in `panel` is ignored, not rejected.**

Ask for `"delinquency_rate"` and you do not get an error. You get a panel that
is missing a column, looks perfectly plausible, and quietly answers a different
question. The SDK validates names against the live catalog for exactly this
reason — it raises a `ValueError` locally before a request is ever sent.

In [ ]:
try:
    silo.panel(["PETR4"], ["delinquency_rate"], freq="month")
except ValueError as exc:
    print("ValueError (raised locally, no request sent):")
    print(" ", exc)

## The two tiers

Anonymous is free and deliberately small — sized for discovery: read the
catalog, check coverage, resolve a name, sample an instrument or two.

Signing in with GitHub at
[silo-bz.vercel.app/signin.html](https://silo-bz.vercel.app/signin.html) raises
the ceilings and unlocks `panel` universe mode (a whole fund family in one paged
call). The token goes in `SILO_TOKEN`, lasts about an hour, and **expires
silently** — you drop back to anonymous limits rather than getting a `401`.

One ceiling does **not** move: rows per response. That is PostgREST's
server-wide `db-max-rows`, identical for every caller.

In [ ]:
limits = silo.limits()
anon, auth = limits["tiers"]["anon"], limits["tiers"]["authenticated"]

rows = [
    {"ceiling": k, "anonymous": anon[k], "signed in": auth[k]}
    for k in ("panel_ids", "panel_universe", "search_funds_rows",
              "option_chain_rows", "option_exercises_rows",
              "fund_holdings_rows", "fidc_sacados_rows",
              "statement_timeout_seconds")
]
rows.append({"ceiling": "rows_per_response",
             "anonymous": limits["rows_per_response"]["value"],
             "signed in": limits["rows_per_response"]["value"]})

print(pd.DataFrame(rows).to_string(index=False))
print()
print("rows_per_response scope:", limits["rows_per_response"]["scope"])
print()
print(f"you are on the {silo.tier!r} tier")

## The three errors

These are not edge cases. You will hit all three, and each one means something
specific. The SDK gives each its own exception class so you do not have to parse
a body.

### 1. `SiloError` — a tier ceiling (`22023`)

You asked for more **ids** than your tier allows. The server names the limit and
what you sent. Nothing is trimmed to fit, because a panel that was silently
shortened to three ids is indistinguishable from one where the fourth fund had
no data.

In [ ]:
from silo_client.client import SiloError, SiloOverCap, SiloTruncated, SiloTimeout

try:
    silo.panel(["PETR4", "VALE3", "ITUB4", "BBAS3"], ["close"], freq="month",
               start="2026-01-01")
except SiloError as exc:
    print(f"{type(exc).__name__}: HTTP {exc.status}")
    print(exc.body)

### 2. `SiloOverCap` — a window larger than one page (`22023`)

Different cause, same SQLSTATE. Your request was **legal** but its result would
be more than one 1,000-row page, so the function **refused** rather than return
a slice that looks whole.

Three functions give you a cursor to walk past it (`panel`, `quote_history`,
`fund_nav`); the rest ask you to narrow the window. Notebook 01 does the walk.

In [ ]:
try:
    silo.panel(["PETR4"], ["close"], freq="day", start="2019-01-01")
except SiloOverCap as exc:
    print(f"{type(exc).__name__}: HTTP {exc.status}")
    print(exc.body)
    print()
    print("hint:", exc.hint)

### 3. `SiloTruncated` — a **view** that was cut (`200`, silently)

The GET views do not refuse. PostgREST cuts them at 1,000 rows, keeps the
**oldest** ones, and answers `200`. There is no error in the body; the only
signal is the `Content-Range` response header.

The SDK asks for `Prefer: count=exact` on every request, compares the count to
what arrived, and raises. **This is the defect the SDK exists to prevent** — and
it is the one you would never notice by eye, because a cut registry looks exactly
like a small registry.

In [ ]:
try:
    silo.view("funds", order="cnpj.asc", select="cnpj,fund_name,entity_type")
except SiloTruncated as exc:
    print(f"{type(exc).__name__}")
    print(f"  returned : {exc.returned:,} rows")
    print(f"  total    : {exc.total:,} rows")
    print(f"  {exc.body}")

Views are the one surface that genuinely pages, so ask for a page explicitly —
or let `view_all` walk it for you. An explicit `limit`/`offset` means you asked
for a page, so the truncation guard stands down.

In [ ]:
page = silo.view("funds", order="cnpj.asc", select="cnpj,fund_name,entity_type",
                 limit=5, offset=0)
pd.DataFrame(page)

### And a fourth you should recognise: `SiloTimeout` (`57014`)

The anonymous role has a **3-second** statement budget (8 seconds signed in).
A query that exceeds it is cancelled and returns nothing — never a partial
result. It is not a server fault and not an outage.

The first call after the database has been idle can be far slower than a warm
one, so a cold `57014` is worth retrying once before concluding anything. When
it persists: narrow the window, ask for fewer ids, or request fewer metrics.

In [ ]:
print(SiloTimeout.__doc__)

## `metric_coverage` — which metrics a family actually files

A null is not evidence of a gap. `fund_nav` returns the same eleven columns for
every fund family, but each family files only some of them, and a null outside
that list is **not applicable** — set by construction, carrying no information.

`metric_coverage()` is the authoritative list of what is genuinely filed, and
over what span. **A `(family, metric)` pair absent from it is one that family
never files** — not one whose data is late.

In [ ]:
mc = pd.DataFrame(silo.metric_coverage())
mc["filed_pct"] = (100 * mc["filed_rows"] / mc["total_rows"]).round(1)
mc

Look at the `fidc` / `delinquency` row. Its `first_period` is **2025-01-31**,
and `filed_pct` is far below 100 — that is not patchy reporting, it is a
**regime break**: CVM's pre-2025 FIDC file had no delinquency field at all.
Notebook 04 works through what that means.

The applicability map itself is in the catalog:

In [ ]:
appl = silo.catalog()["applicability"]["fund_nav"]
print(appl["rule"], "\n")
for family, cols in appl["columns_by_family"].items():
    print(f"  {family:7s} files: {', '.join(cols)}")
print()
print("period convention (this is NOT uniform):")
for family, conv in appl["period_convention"].items():
    print(f"  {family:7s} {conv}")

## Where to go next

| Notebook | Question |
| --- | --- |
| `01_market_tape` | What has PETR4 done since 2019, and how do I get more than 1,000 sessions? |
| `02_fund_nav_flows` | Did this fund's investors put money in or take it out? |
| `03_fund_holdings` | Which funds hold PETR4, and who holds Petrobras debentures? |
| `04_fidc_credit` | How concentrated and how delinquent is this receivables fund? |
| `05_listed_companies` | What did Petrobras earn last quarter, without double-counting it? |
| `06_anbima_classes` | Where did the industry's money go this year, by ANBIMA class? |
| `07_derivatives` | What does the front-month PETR4 option chain look like? |
| `08_short_interest` | Who is short, who is lending, and who is buying? |

---

## The rules this notebook obeyed

* **Nothing was filled.** No forward-fill, no interpolation, no carried-forward
  last observation. A gap in a chart is a gap in the filings.
* **Every caveat was printed beside its number** — `coverage().notes`,
  `catalog().regime_breaks`, `catalog().applicability`, `float_basis` — rather
  than left in a docstring somewhere.
* **Freshness came from `coverage()`**, called before anything was claimed.

The contract these rules come from is
[Conventions & limits](https://octo-98895abd.mintlify.site/api-docs/conventions),
and its machine-readable twin is `POST /rpc/catalog`.